# AoC 2024 Day 15 — Warehouse Woes

**Python — sequential state mutation (and why Spark loses)**

Puzzle: <https://adventofcode.com/2024/day/15>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A warehouse map plus a long string of attempted robot moves.

The map uses `#` for walls, `O` for boxes, `@` for the robot, `.` for floor. The moves are `^ v < >`, wrapped over several lines in the input but meant as one continuous sequence.

When the robot moves into a box, it pushes it — and any unbroken run of boxes behind it — one step along. If **anything** in that run would be pushed into a wall, nothing moves at all, including the robot.

- **Part 1** — run every move in order, then sum the **GPS coordinate** of each box: `100 × (distance from the top edge) + (distance from the left edge)`, measured against the full map, walls included.

## The approach

This is the day the repo uses Spark for nothing. `aoc_spark/y2024/day15.py` still takes a `SparkSession` to keep the interface uniform, and then never touches it. That is a deliberate answer, not a shortcut, and it is worth being precise about why.

**The sequential dependency is total.** Move *k* reads the grid that move *k−1* produced. Not a summary of it — the grid. Whether `>` does anything depends on how far the run of boxes to the robot's right extends *right now*, which depends on every push that came before. There is no move you can evaluate without first evaluating all 20000 of its predecessors.

Compare **day 3**, which looks equally sequential and is not. There the carried state was a single boolean, the transition was "keep the last non-null", and that is associative — so a window function computes all of it in one pass. The scan trick works whenever the carried state is small and the transition composes.

Here the carried state is the entire 50×50 grid and the transition is a ray walk. Composing two moves without a grid to walk means representing "what this move does to an arbitrary grid", which is not smaller than the grid — so there is nothing to prefix-scan, and no window, join, or aggregate recovers the parallelism.

**What a Spark version would actually cost.** The only faithful shape is one job per move: 20000 rounds, each reading the box positions, walking the ray, writing the updated set, and hitting an action so the driver can decide the next move. At even ~50 ms of Spark Connect round-trip per round — optimistic for a plan that includes a join — that is roughly **17 minutes**. The Python simulation finishes the same 20000 moves in a few milliseconds.

And there is nothing to win back at scale, because there is no scale. The grid is 2500 cells with 623 boxes on it; that is one partition, and per move the real work is reading a handful of adjacent cells. **Spark's unit of work is larger than the entire problem.** The right call is a mutable `list[list[str]]` and a `while` loop.

The one thing worth stealing from the implementation is the push itself. A run of *n* boxes shifting one step leaves every interior box exactly where it was — only the ends change. So instead of moving *n* boxes, teleport the near box to the far gap: write `O` at the first free cell past the run, and `.` where the robot lands. Two writes regardless of how long the run is.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day15

spark = get_spark('aoc-2024-day15')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '##########\n#..O..O.O#\n#......O.#\n#.OO..O.O#\n#..O@..O.#\n#O#..O...#\n#O..O..O.#\n#.OO.O.OO#\n#....O...#\n##########\n\n<vv>^<v^>v>^vv^v>v<>v^v<v<^vv<<<^><<><>>v<vvv<>^v^>^<<<><<v<<<v^vv^v>^\nvvv<<^>^v^^><<>>><>^<<><^vv^^<>vvv<>><^^v>^>vv<>v<<<<v<^v>^<^^>>>^<v<v\n><>vv>v^v^<>><>>>><^^>vv>v<^^^>>v^v^<^^>v^^>v^<^v>v<>>v^v^<v>v^^<^^vv<\n<<v<^>>^^^^>>>v^<>vvv^><v<<<>^^^vv^<vvv>^>v<^^^^v<>^>vvvv><>>v^<<^^^^^\n^><^><>>><>^^<<^^v>>><^<v>^<vv>>v>>>^v><>^v><<<<v>>v<v<v>vvv>^<><<>^><\n^>><>^v<><^vvv<^^<><v<<<<<><^v<<<><<<^^<v<^^^><^>>^<v^><<<^>>^v<v^v<v^\n>^>>^v>vv>^<<^v<>><<><<v<<v><>v<^vv<<<>^^v^>^^>>><<^v>>v^v><^^>>^<>vv^\n<><^^>^^^<><vvvvv^v<v<<>^v<v>v<<^><<><<><<<^^<<<^<<>><<><^^^>^^<>^>v<>\n^^>vv<^v^v<vv>^<><v<^v>^^^>>>^^vvv^>vvv<>>>^<^>>>>>^<<^v>^vvv<>^<><<v>\nv^^>>><<^^<>>^v^<v^vv<>v^<<>^<^v^v><^<<<><<^<v><v<>vv>>v><v^<vv<>v^<<^\n'

print('part 1:', day15.part1(spark, EXAMPLE), '(expected 10092)')

### Why there is nothing to parallelise

The prefix table shows the answer wandering with no relationship to where it lands. The last two lines are the point: the *same multiset of moves* in a different order gives a different answer, so no reordering, batching, or partitioning of the move list is available.

In [ ]:
grid, moves = day15.parse(EXAMPLE)
boxes = sum(row.count('O') for row in grid)
print(f'{len(grid)}x{len(grid[0])} grid, {boxes} boxes, {len(moves)} moves')
print('moves start:', moves[:48])


def replay(grid, moves):
    """The same simulation, instrumented -- returns (blocked, gps)."""
    grid = [row[:] for row in grid]
    r, c = next(
        (r, c) for r, row in enumerate(grid) for c, ch in enumerate(row) if ch == '@'
    )
    blocked = 0
    for move in moves:
        dr, dc = day15.DELTAS[move]
        nr, nc = r + dr, c + dc
        er, ec = nr, nc
        while grid[er][ec] == 'O':
            er, ec = er + dr, ec + dc
        if grid[er][ec] == '#':
            blocked += 1
            continue
        grid[er][ec] = 'O' if (er, ec) != (nr, nc) else '.'
        grid[nr][nc] = '@'
        grid[r][c] = '.'
        r, c = nr, nc
    gps = sum(
        100 * r + c
        for r, row in enumerate(grid)
        for c, ch in enumerate(row)
        if ch == 'O'
    )
    return blocked, gps


# The answer only exists at the end of the sequence -- no prefix predicts it.
for n in (0, 1, 10, 100, 400, len(moves)):
    blocked, gps = replay(grid, moves[:n])
    print(f'after {n:>4} moves: {blocked:>4} blocked, GPS sum {gps}')

# Same moves, different order: the state dependency, priced.
half = len(moves) // 2
shuffled = moves[half:] + moves[:half]
print('halves swapped ->', replay(grid, shuffled)[1])
print('reversed       ->', replay(grid, moves[::-1])[1])

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 15)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day15.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day15 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **Moves are wrapped across several lines** purely for the puzzle page's benefit. `"".join(move_block.split())` rejoins them; splitting on newlines and treating each line as a run would still work here, but treating a `\n` as a move raises a `KeyError` on `DELTAS`.
- `data.strip("\n")` — **not** `.strip()`. A bare `strip()` would eat leading spaces, and while this input has none, the grid block is whitespace-significant.
- The push shortcut (`grid[er][ec] = "O"`) relies on the run between the robot and the gap being **entirely boxes**, which the `while` loop guarantees, and on the near cell being overwritten by `@` immediately after. Reorder those two writes and a single-box push corrupts the grid.
- The `(er, ec) != (nr, nc)` test distinguishes *pushed boxes* from *walked into empty space*. Without it, stepping into a `.` writes a phantom `O` behind the robot.
- **The border is assumed to be solid `#`.** The ray walk has no bounds check at all — it relies on hitting a wall before running off the grid. That is true of every AoC input for this day and is the assumption most likely to bite anyone feeding it a hand-made grid.
- GPS uses the **full map** coordinates including the wall border, so the top-left wall is `(0, 0)` and no offset is subtracted anywhere.
- Exactly one `@` is assumed; `next(...)` on the generator takes the first and would raise `StopIteration` on a grid with none.
- The puzzle publishes **two** examples: a small 8×8 one worth 2028 and a large 10×10 one with 700 moves worth **10092**. The repo uses the large one — checking against the wrong published answer is the easiest way to lose an hour here.